# Day 31 — Exploratory Data Analysis (Nigerian Retail Sales Analyzer)

This notebook covers Gross Merchandise Value (GMV) Aggregations, Regional Hub Performance, Temporal Trends, and Product-Level Insights

## Validation goals
- Load the pristine, cleaned dataset (`cleaned_retail_sales.csv`) and engineer the total revenue feature (`price * quantity`)
- Analyze Gross Merchandise Value (GMV), unit turnover, average unit prices, and transaction counts across product categories using `.groupby()` and `.agg()`
- Compare commercial activity across Nigerian regional market hubs (Computer Village Lagos, Wuse Market Abuja, Onitsha Main Market Anambra, Balogun Market Lagos, and Kantin Kwari Kano)
- Evaluate monthly revenue velocity and transaction volume cycles using period resampling (`dt.to_period("M")`)
- Extract top-performing inventory items by revenue contribution

With a clean, validated dataset saved in `cleaned_retail_sales.csv`, Day 31 shifts focus to Exploratory Data Analysis (EDA). EDA is the investigative phase where we query, group, and slice our data to uncover hidden patterns, evaluate regional commercial performance, analyze category demand, and track temporal sales velocity across major Nigerian market hubs.

### Core Analysis Objectives

1. Revenue & Volume Breakdown by Category: Identify which product categories (Electronics, Groceries, Apparel, Home Appliances) drive the highest gross merchandise value (GMV) and unit turnover.

2. Regional Market Performance: Compare commercial activity across markets (e.g., Balogun Market Lagos, Wuse Market Abuja, Kantin Kwari Kano) to determine top revenue-generating hubs.

3. Temporal Trends: Analyze sales distributions across months and operational cycles.

4. Supplier Contributions: Assess vendor-level transaction distribution.


### Implementation Script:

In [1]:
import pandas as pd
import numpy as np

# 1. Load Cleaned Dataset
df = pd.read_csv("cleaned_retail_sales.csv", parse_dates=["date"])

# Feature Engineering: Compute total transaction revenue
df["total_revenue"] = df["price"] * df["quantity"]

print("--- Dataset Overview ---")
print(f"Total Transactions: {len(df):,}")
print(f"Total Gross Revenue: ₦{df['total_revenue'].sum():,.2f}\n")


# ==========================================
# 2. Category & Product Performance Analysis
# ==========================================
category_summary = df.groupby("category", as_index=False).agg(
    total_revenue=("total_revenue", "sum"),
    total_units_sold=("quantity", "sum"),
    avg_price=("price", "mean"),
    transaction_count=("product", "count")
).sort_values(by="total_revenue", ascending=False)

print("--- Performance by Category ---")
print(category_summary.to_string(index=False))


# ==========================================
# 3. Regional Market Hub Analysis
# ==========================================
location_summary = df.groupby("location", as_index=False).agg(
    total_revenue=("total_revenue", "sum"),
    total_units=("quantity", "sum"),
    avg_basket_value=("total_revenue", "mean")
).sort_values(by="total_revenue", ascending=False)

print("\n--- Performance by Market Location ---")
print(location_summary.to_string(index=False))


# ==========================================
# 4. Temporal Trend Analysis (Monthly Aggregation)
# ==========================================
df["sales_month"] = df["date"].dt.to_period("M")
monthly_trend = df.groupby("sales_month", as_index=False).agg(
    monthly_revenue=("total_revenue", "sum"),
    monthly_transactions=("product", "count")
)

print("\n--- Monthly Revenue Velocity ---")
print(monthly_trend.to_string(index=False))


# ==========================================
# 5. Top Performing Products
# ==========================================
top_products = df.groupby("product", as_index=False).agg(
    product_revenue=("total_revenue", "sum"),
    units_sold=("quantity", "sum")
).sort_values(by="product_revenue", ascending=False).head(5)

print("\n--- Top 5 Products by Revenue ---")
print(top_products.to_string(index=False))

--- Dataset Overview ---
Total Transactions: 5,000
Total Gross Revenue: ₦8,492,075,611.59

--- Performance by Category ---
       category  total_revenue  total_units_sold     avg_price  transaction_count
      Groceries   2.234643e+09              9941 223611.550339               1307
Home Appliances   2.121333e+09              9400 228591.393447               1226
    Electronics   2.075016e+09              9045 227867.832342               1219
        Apparel   2.061083e+09              9369 222752.016073               1248

--- Performance by Market Location ---
                    location  total_revenue  total_units  avg_basket_value
     Computer Village, Lagos   1.736963e+09         7707      1.728322e+06
          Wuse Market, Abuja   1.701671e+09         7457      1.770729e+06
Onitsha Main Market, Anambra   1.670976e+09         7455      1.643044e+06
       Balogun Market, Lagos   1.669301e+09         7636      1.649507e+06
          Kantin Kwari, Kano   1.659705e+09         